### Face 2: ingenieria de caracteristicas.

| Vamos a extraer caracteristicas por cada registro

Importamos librerias

In [20]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
tqdm.pandas()

In [21]:
df = pd.read_csv('dataset_optimizado.csv')
def crear_features_pro(df):
    df = df.copy()
    
    df['Hora_Inicio'] = pd.to_datetime(
        df['Hora_Inicio'],
        format='%H:%M:%S'
    )
    df['Hora_Minutos'] = df['Hora_Inicio'].dt.hour * 60 + df['Hora_Inicio'].dt.minute
    
   
    df['Periodo_Dia'] = pd.cut(df['Hora_Minutos'], 
                                bins=[0, 360, 720, 1080, 1440], 
                                labels=[0, 1, 2, 3]) 
    
    df = df.sort_values(['Direccion', 'Hora_Inicio'])
    
    group = df.groupby('Direccion')
    
    df['Total_Vehiculos_lag1'] = group['Total_Vehiculos'].shift(1)
    df['Ocupacion_lag1'] = group['Ocupacion_Espacial_%'].shift(1)
    
    df['Media_Movil_3ciclos'] = group['Total_Vehiculos'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    
    df['Tendencia_Vehiculos'] = group['Total_Vehiculos'].diff()
    
    df['Capacidad_Teorica'] = df['move_time_s'] / df['Tiempo_Medio_s'].replace(0, 1)
    df['Saturacion_Actual'] = df['Total_Vehiculos'] / df['Capacidad_Teorica'].replace(0, 1)

    df = df.fillna(0)
    
    return df

df = crear_features_pro(df)
display(df.head())
df.to_csv('completo_clusters_con_features.csv', index=False)

,Total_Vehiculos,Tiempo_Medio_s,Ocupacion_Espacial_%,Hora_Inicio,Hora_Fin,Dia_Semana,Direccion,move_time_s,cluster,tiempo_optimo_s,Hora_Minutos,Periodo_Dia,Total_Vehiculos_lag1,Ocupacion_lag1,Media_Movil_3ciclos,Tendencia_Vehiculos,Capacidad_Teorica,Saturacion_Actual
1523,7,1.67,17.71,1900-01-01 08:00:09,08:00:42,2,1,33,0,35,480,1,0.0,0.00,7.000000,0.0,19.760479,0.354242
2284,9,3.68,21.95,1900-01-01 08:00:20,08:00:53,3,1,33,0,35,480,1,7.0,17.71,8.000000,2.0,8.967391,1.003636
3,7,4.23,19.05,1900-01-01 08:00:29,08:01:02,4,1,33,0,35,480,1,9.0,21.95,7.666667,-2.0,7.801418,0.897273
764,7,2.00,16.25,1900-01-01 08:01:32,08:02:05,1,1,33,0,35,481,1,7.0,19.05,7.666667,0.0,16.500000,0.424242
3045,9,3.03,23.80,1900-01-01 08:01:43,08:02:16,5,1,33,0,35,481,1,7.0,16.25,7.666667,2.0,10.891089,0.826364


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.ticker import FuncFormatter

# Función para formatear el eje X (de minutos 0-1440 a HH:MM)
def formato_tiempo(x, pos):
    horas = int(x // 60)
    minutos = int(x % 60)
    return f"{horas:02d}:{minutos:02d}"

def visualizar_resultados_con_stats(test_data, y_test_pred, fijo_verde=30, col_hora='Hora'):
    """
    Genera gráficos comparativos por dirección e incluye las estadísticas en la imagen.
    Asume que 'col_hora' está en minutos (0 a 1440).
    """
    df = test_data.copy()
    df['Verde_Modelo'] = y_test_pred
    df['Verde_Fijo'] = fijo_verde
    
    # Aseguramos que esté ordenado por hora para que la línea del gráfico sea continua
    df = df.sort_values(by=col_hora)
    
    direcciones = sorted(df['Direccion'].unique())
    n_dirs = len(direcciones)
    
    # Configurar la figura (2 columnas y filas necesarias)
    cols = 2
    rows = (n_dirs + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(15, 6 * rows), constrained_layout=True)
    axes = axes.flatten()
    
    formatter = FuncFormatter(formato_tiempo)

    for i, direccion in enumerate(direcciones):
        ax = axes[i]
        datos = df[df['Direccion'] == direccion]
        
        # --- 1. DATOS ESTADÍSTICOS (Tu lógica original) ---
        promedio_modelo = datos['Verde_Modelo'].mean()
        diferencia = promedio_modelo - fijo_verde
        
        texto_stats = (
            f"DIRECCIÓN {direccion}\n"
            f"----------------------\n"
            f"Modelo ML : {promedio_modelo:.1f} s\n"
            f"Fijo      : {fijo_verde:.1f} s\n"
            f"Diferencia: {diferencia:+.1f} s"
        )
        
        # --- 2. GRAFICAR ---
        # Línea del Modelo
        sns.lineplot(data=datos, x=col_hora, y='Verde_Modelo', ax=ax, 
                     label='Predicción Modelo', color='#2ca02c', alpha=0.7)
        
        # Línea Fija
        ax.axhline(y=fijo_verde, color='red', linestyle='--', linewidth=2, label=f'Fijo ({fijo_verde}s)')
        
        # --- 3. FORMATO ---
        ax.set_title(f"Comportamiento Semáforo - Dirección {direccion}", fontsize=14)
        ax.set_xlabel("Hora del Día")
        ax.set_ylabel("Tiempo de Verde (segundos)")
        ax.xaxis.set_major_formatter(formatter) # Aplicar formato HH:MM
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right')
        
        # --- 4. AÑADIR CAJA DE TEXTO CON DATOS ---
        # Colocamos la caja en la esquina superior izquierda
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
        ax.text(0.02, 0.95, texto_stats, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=props)

    # Ocultar ejes vacíos si hay número impar de direcciones
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
        
    plt.show()

# --- EJECUCIÓN ---
# Asegúrate de cambiar 'Hora' por el nombre real de tu columna de tiempo si es diferente
visualizar_resultados_con_stats(
    prueba_data, 
    y_prueba_pred, 
    fijo_verde=30, 
    col_hora='Hora'  # <--- Nombre de tu columna de tiempo (0-1440)
)

NameError: name 'prueba_data' is not defined